# NullVector Progress Notebook

major-changes-v2 / Phase F-J continuity / Phase 4 / Developer Experience

Purpose: manually verify the new `NullVectorClient`, local catalog flow, and installed CLI surface.

This notebook assumes the repository fixtures are present and writes only into `.artifacts/progress-phase4-client-cli`.

### Environment

This notebook uses a local markdown fixture and `typer.testing.CliRunner` so it can exercise the client and CLI deterministically without requiring external services.

In [ ]:
# environment setup
import json
import shutil
from pathlib import Path

artifact_dir = Path('.artifacts/progress-phase4-client-cli').resolve()
if artifact_dir.exists():
    shutil.rmtree(artifact_dir)
artifact_dir.mkdir(parents=True, exist_ok=True)
artifact_dir

In [ ]:
# imports
from typer.testing import CliRunner

from nullvector import NullVectorClient
from nullvector.cli import app


In [ ]:
# configuration
client_workspace = artifact_dir / 'client-workspace'
cli_workspace = artifact_dir / 'cli-workspace'
source_path = artifact_dir / 'phase4-client-cli.md'
source_path.write_text(
    '\n'.join((
        '# Overview',
        'NullVector Phase 4 adds a queryable client facade.',
        '',
        '## Findings',
        'The installed CLI now builds retrieval artifacts by default.',
        '',
        '### Appendix',
        'Client ask resolves the document through the local catalog.',
    )),
    encoding='utf-8',
)
runner = CliRunner()
client_workspace, cli_workspace, source_path

In [ ]:
# execution
client = NullVectorClient(storage_path=client_workspace)
client_result = client.ingest(source_path, source_kind='markdown', preset='general_document')
client_hits = client.search('retrieval artifacts', document_id=client_result.document_id)
client_answer = client.ask('What does the installed CLI build by default?', document_id=client_result.document_id)

ingest_cli = runner.invoke(
    app,
    [
        'ingest',
        str(source_path),
        '--source-kind',
        'markdown',
        '--storage-path',
        str(cli_workspace),
        '--preset',
        'general_document',
    ],
)
assert ingest_cli.exit_code == 0, ingest_cli.stderr
ingest_payload = json.loads(ingest_cli.stdout)
ask_cli = runner.invoke(
    app,
    [
        'ask',
        'What does the installed CLI build by default?',
        '--document-id',
        ingest_payload['document_id'],
        '--storage-path',
        str(cli_workspace),
    ],
)
assert ask_cli.exit_code == 0, ask_cli.stderr
ask_payload = json.loads(ask_cli.stdout)

catalog_payload = json.loads((client_workspace / '.nullvector' / 'catalog.json').read_text(encoding='utf-8'))

phase4_results = {
    'client': {
        'document_id': client_result.document_id,
        'retrieval_manifest_path': client_result.retrieval_manifest_path,
        'hit_count': len(client_hits),
        'answer': client_answer.answer,
    },
    'cli': {
        'ingest_exit_code': ingest_cli.exit_code,
        'ask_exit_code': ask_cli.exit_code,
        'document_id': ingest_payload['document_id'],
        'answer': ask_payload['answer'],
    },
    'catalog_documents': tuple(sorted(catalog_payload['documents'].keys())),
}
phase4_results

In [ ]:
# inspect results
print(json.dumps(phase4_results, indent=2, sort_keys=True))

### Known Limitations

This Phase 4 notebook intentionally uses separate client and CLI workspaces so the smoke path stays deterministic even when both flows derive the same document fingerprint and default run ids. It does not exercise external LLM providers or Postgres-backed client storage in the notebook path.